# R&D

In [41]:
# Loading Packages
import pandas as pd
import numpy as np
import random
import os
import torch
import faiss
import json
import re
import shutil
import random
from tqdm import tqdm
from torch import Tensor
from dotenv import load_dotenv
from huggingface_hub import login
from torch.utils.data import DataLoader
from beir.datasets.data_loader import GenericDataLoader
from transformers import AutoTokenizer, logging, AutoModel, AutoModelForCausalLM
logging.set_verbosity_error()

In [47]:
### Pre ambles ###
save_name='_syn_qag_1'
llm_temp=0.1
max_token=256
test=True

In [43]:
### Loading data ###
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 8841823/8841823 [01:19<00:00, 110852.43it/s]


In [44]:
# Gathering query data
query_info = [(key, value) for key, value in queries.items()]

In [45]:
### Loading LM Model ###
# Authenticating Token
load_dotenv('/work/mbouthil/MMATH-CM-Research-Project/token.env')
token = os.getenv('HUGGINGFACE_TOKEN')

# Loading model and Tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer =  AutoTokenizer.from_pretrained(model_name, token=token)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    token=token
)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 466.52it/s, Materializing param=model.norm.weight]                              


In [48]:
### Sysyem Prompts ###
cot_prompt='''
You are a helpful AI assistant. You are to follow the following instructions:

You will be given a query. Your task is to think about how you would produce a new query asking the same underlying question.

Proivide your thoughts:
'''

creation_prompt=f'''
You are a helpful AI assistant. Your task is to create a new question from the provided query, asking the same underlying infromation. 

Provide only your new question.  

Moreover, consider the following:
'''

judge_prompt='''
You are a helpful AI assistant. 

Your task is to judge whether both provided queries pose the same underlying question. 

Ensure that your answer contains TRUE or FALSE. 
'''

In [49]:
### LLM function ###
def llm_pass(
        messages:list[list[dict]],
        padding:bool=True,
        truncation:bool=True,
        max_tokens:int=max_token, 
        temp:float=llm_temp,
        top_p:float=0.9,
) -> list[str]:

    prompts = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=padding,
        truncation=truncation
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temp,
            top_p=top_p,
            do_sample=True
        )

    responses = []
    for i in range(len(messages)):
        gen = outputs[i][inputs["input_ids"].shape[1]:]
        responses.append(tokenizer.decode(gen, skip_special_tokens=True))

    return responses

In [50]:
def batch_splits(item:list, batch_size:int=64):

    for i in range(0, len(item), batch_size):
        yield item[i:i + batch_size]

batches = batch_splits(query_info)

In [51]:
print(len(queries))

502939


In [ ]:
for batch in batches:

    max_id = max([int(key) for key in queries.keys()])
    q_ids, query_texts = zip(*batch)

    # Step 1
    cot_messages = [
        [
            {"role": "system", "content": cot_prompt},
            {"role": "user", "content": query}
        ]
        for query in query_texts
    ]
    cot_responses = llm_pass(cot_messages)
    cot_responses = [response[11:] for response in cot_responses]

    # Step 2
    creation_messages = [
        [
            {"role": "system", "content": creation_prompt + cot_responses[i]},
            {"role": "user", "content": query}
        ]
        for i, query in enumerate(query_texts)
    ]
    syn_queries = llm_pass(creation_messages)
    syn_queries = [query[11:] for query in syn_queries]

    # Step 3
    judge_messages = [
        [
            {"role": "system", "content": judge_prompt},
            {"role": "user", "content": 
             str('"' + query_texts[i] + '"\n\nand \n' + syn_queries[i])
             }
        ]
        for i in range(len(query_texts))
    ]
    verdicts = llm_pass(judge_messages)
    verdicts = [1 if 'TRUE' in answer else 0 for answer in verdicts]

    filtered = [(id, query) for id, query, verdict in zip(q_ids, syn_queries, verdicts) if verdict == 1]
    q_ids, new_queries = [list(item) for item in zip(*filtered)]

    for i, q_id in enumerate(q_ids):
        new_id = max_id + i
        qrels[new_id] = qrels[q_id]
        queries[new_id] = new_queries[i]

    print('Batch Complted')
    if test == True:
        break

In [ ]:
print(len(queries))

502941
